# Finance Project 1

In [25]:
import wrds
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.regression.linear_model import OLS
from statsmodels.tools.tools import add_constant
import plotly.graph_objects as go
from pathlib import Path
import pickle

def connect_wharton(username="zkg232"):
    """Connect to WRDS PostgreSQL (our prior method: explicit username, uses .pgpass for password)."""
    return wrds.Connection(wrds_username=username)

db = connect_wharton()

# Cache for WRDS data to avoid re-running queries
CACHE_DIR = Path('cache')
CACHE_DIR.mkdir(exist_ok=True)

def load_or_fetch(name, loader):
    path = CACHE_DIR / f"{name}.pkl"
    if path.exists():
        with open(path, 'rb') as f:
            return pickle.load(f)
    data = loader()
    with open(path, 'wb') as f:
        pickle.dump(data, f)
    return data

Loading library list...
Done


## Compustat Query - Get R&D Data and Link to CRSP

In [26]:
comp_sql = """
SELECT
    a.gvkey, 
    a.datadate, 
    a.xrd,
    b.lpermno AS permno
FROM
    comp.funda a
JOIN
    crsp.ccmxpf_linktable b ON a.gvkey = b.gvkey
WHERE
    a.indfmt = 'INDL'
    AND a.datafmt = 'STD'
    AND a.popsrc = 'D'
    AND a.consol = 'C'
    AND a.fyear BETWEEN 1978 AND 2022
    AND a.fic = 'USA'
    AND a.curcd = 'USD'
    AND a.exchg BETWEEN 11 AND 19
    AND (a.sich < 6000 OR a.sich > 6999 OR a.sich IS NULL)
    AND (a.sich != 2834 OR a.sich IS NULL)
    AND b.linktype IN ('LU', 'LC')
    AND b.linkprim IN ('P', 'C')
    AND b.usedflag = 1
    AND (b.linkdt <= a.datadate OR b.linkdt IS NULL)
    AND (b.linkenddt >= a.datadate OR b.linkenddt IS NULL)
"""
comp = load_or_fetch('comp', lambda: db.raw_sql(comp_sql))

## CRSP Query - Monthly Returns with Delisting Adjustment

In [27]:
comp.sort_values('datadate')

,gvkey,datadate,xrd,permno
72759,010063,1978-06-30,<NA>,27254.0
42297,006251,1978-06-30,<NA>,46544.0
42198,006228,1978-06-30,1.858,46296.0
42012,006215,1978-06-30,<NA>,46229.0
41982,006214,1978-06-30,0.807,47984.0
...,...,...,...,...
189294,155336,2023-05-31,<NA>,89783.0
88962,012142,2023-05-31,8623.0,10104.0
155136,031846,2023-05-31,0.0,81655.0
68314,009466,2023-05-31,<NA>,77354.0


In [28]:
crsp_sql = """
SELECT 
    a.permno,
    a.date,
    a.ret,
    a.prc,
    a.shrout,
    c.dlret
FROM 
    crsp.msf a
INNER JOIN crsp.msenames b ON a.permno = b.permno
    AND a.date BETWEEN b.namedt AND b.nameendt
LEFT JOIN crsp.msedelist c ON a.permno = c.permno
    AND date_trunc('month', a.date) = date_trunc('month', c.dlstdt)
WHERE 
    a.date BETWEEN '1970-01-01' AND '2023-06-30'
    AND a.ret IS NOT NULL
    AND a.ret >= -1
    AND b.shrcd IN (10, 11)
    AND (b.siccd < 6000 OR b.siccd > 6999 OR b.siccd IS NULL)
    AND (b.siccd != 2834 OR b.siccd IS NULL)
    AND b.exchcd IN (1, 2, 3)
"""
crsp = load_or_fetch('crsp', lambda: db.raw_sql(crsp_sql))

In [29]:
crsp.sort_values('date')


,permno,date,ret,prc,shrout,dlret
401775,18243,1970-01-30,-0.071895,-17.75,926.0,<NA>
446542,19879,1970-01-30,0.147541,26.25,2650.0,<NA>
213436,13100,1970-01-30,-0.060465,25.25,15107.0,<NA>
470125,44089,1970-01-30,-0.145833,5.125,604.0,<NA>
376263,39917,1970-01-30,-0.133951,34.875,59572.0,<NA>
...,...,...,...,...,...,...
119200,83885,2023-06-30,-0.023454,4.58,187620.0,<NA>
414577,18560,2023-06-30,0.075463,22.66,7522.0,<NA>
381958,89453,2023-06-30,0.091555,13.83,16121.0,<NA>
436914,19462,2023-06-30,-0.030804,0.258,39429.0,<NA>


## Adjust for Delisting Returns

In [30]:
crsp['dlret'] = pd.to_numeric(crsp['dlret'], errors='coerce')
crsp['ret'] = np.where(crsp['dlret'].notna(),
                       (1 + crsp['ret']) * (1 + crsp['dlret']) - 1,
                       crsp['ret'])

## Market Return and Risk-Free Rate

In [31]:
mkt = load_or_fetch('mkt', lambda: db.raw_sql("""
    SELECT date, vwretd, ewretd
    FROM crsp.msi
    WHERE date BETWEEN '1970-01-01' AND '2023-06-30'
"""))

rf = load_or_fetch('rf', lambda: db.raw_sql("""
    SELECT date, rf
    FROM ff.factors_monthly
    WHERE date BETWEEN '1970-01-01' AND '2023-06-30'
"""))

In [45]:
rf

,date,rf
0,1970-01-01,0.006
1,1970-02-01,0.0062
2,1970-03-01,0.0057
3,1970-04-01,0.005
4,1970-05-01,0.0053
...,...,...
637,2023-02-01,0.0034
638,2023-03-01,0.0036
639,2023-04-01,0.0035
640,2023-05-01,0.0036


## Create R&D Flag and Lag It

In [32]:
comp['xrd'] = pd.to_numeric(comp['xrd'], errors='coerce')
comp['has_rd'] = ((comp['xrd'].notna()) & (comp['xrd'] > 0)).astype(int)
comp = comp.sort_values(['permno', 'datadate'])
comp['rd_flag_lag1'] = comp.groupby('permno')['has_rd'].shift(1)

## 4-month (120-day) reporting lag — available_date

In [33]:
# Monthly formation: use fiscal data only from available_date = datadate + 120 days (4 months)
comp['datadate'] = pd.to_datetime(comp['datadate'])
comp['available_date'] = comp['datadate'] + pd.Timedelta(days=120)
comp['available_date'] = comp['available_date'].dt.to_period('M').dt.to_timestamp()  # floor to month for joins
comp = comp.drop_duplicates(subset=['permno', 'datadate'], keep='last')

crsp['date'] = pd.to_datetime(crsp['date'])
crsp['date'] = crsp['date'].dt.to_period('M').dt.to_timestamp()  # floor to month for joins

## Market Cap - Use Lagged for Value Weighting

In [34]:
crsp['ret'] = pd.to_numeric(crsp['ret'], errors='coerce')
crsp['prc'] = pd.to_numeric(crsp['prc'], errors='coerce')
crsp['shrout'] = pd.to_numeric(crsp['shrout'], errors='coerce')
crsp['mktcap'] = crsp['prc'].abs() * crsp['shrout']
crsp = crsp.sort_values(['permno', 'date'])
crsp['mktcap_lag'] = crsp.groupby('permno')['mktcap'].shift(1)

## Merge Datasets (monthly formation: reprex-style merge + date filter + keep last)

In [35]:
# Reprex-style merge (no merge_asof): merge on permno, filter date >= available_date, keep last per (permno, date)
comp_for_merge = comp[['permno', 'available_date', 'rd_flag_lag1']].copy()
comp_for_merge = comp_for_merge.dropna(subset=['rd_flag_lag1'])
comp_for_merge['permno'] = pd.to_numeric(comp_for_merge['permno'], errors='coerce').astype('int64')
comp_for_merge = comp_for_merge.dropna(subset=['permno'])
comp_for_merge['available_date'] = pd.to_datetime(comp_for_merge['available_date']).dt.to_period('M').dt.to_timestamp()

crsp_dates = crsp[['permno', 'date']].copy()
crsp_dates['permno'] = pd.to_numeric(crsp_dates['permno'], errors='coerce').astype('int64')
crsp_dates['date'] = pd.to_datetime(crsp_dates['date']).dt.to_period('M').dt.to_timestamp()
crsp_dates = crsp_dates.dropna().drop_duplicates()

merged = crsp_dates.merge(comp_for_merge, on='permno', how='inner')
merged = merged[merged['date'] >= merged['available_date']]
merged = merged.sort_values(['permno', 'date', 'available_date'])
merged = merged.drop_duplicates(subset=['permno', 'date'], keep='last')

crsp_cols = ['permno', 'date', 'ret', 'prc', 'mktcap', 'mktcap_lag']
merged = merged.merge(crsp[crsp_cols], on=['permno', 'date'], how='inner')
merged = merged[(merged['available_date'].dt.year >= 1980) & (merged['available_date'].dt.year <= 2022)]
merged = merged.dropna(subset=['ret', 'mktcap_lag'])
merged = merged[merged['mktcap_lag'] > 0]

## Analysis Data + Investability Flag (Our Conventions)

In [36]:

# Use Nick's returns and prices (ret, prc) and his R&D (rd_flag_lag1) directly
analysis_data = merged[['permno', 'date', 'ret', 'prc', 'mktcap', 'mktcap_lag', 'rd_flag_lag1']].copy()
analysis_data['research_not'] = np.where(analysis_data['rd_flag_lag1'].astype(int) == 1, 'With_XRD', 'Without_XRD')

# Investability: size >= $15M and has prior 12 prices (using Nick's prc)
analysis_data = analysis_data.sort_values(['permno', 'date'])
prior_12_prc_count = (
    analysis_data.groupby('permno')['prc']
    .transform(lambda x: x.shift(1).rolling(12, min_periods=12).count())
)
analysis_data['has_prior_12_prices'] = (prior_12_prc_count >= 12)
size_ok = (analysis_data['mktcap_lag'] >= 15e6) & analysis_data['mktcap_lag'].notna()
analysis_data['investability'] = np.where(size_ok & analysis_data['has_prior_12_prices'], 'Yes', 'No')
analysis_data['portfolio_id'] = analysis_data['research_not'] + '_' + analysis_data['investability']


In [37]:
analysis_data.sort_values('date')

,permno,date,ret,prc,mktcap,mktcap_lag,rd_flag_lag1,research_not,has_prior_12_prices,investability,portfolio_id
279650,18905,1980-01-01,0.000000,13.0,47749.0,47749.0,0.0,Without_XRD,False,No,Without_XRD_No
865483,64427,1980-01-01,0.133333,-1.0625,2713.625,2394.375,0.0,Without_XRD,False,No,Without_XRD_No
380182,25849,1980-01-01,0.116071,15.625,54609.375,48930.0,1.0,With_XRD,False,No,With_XRD_No
703644,52804,1980-01-01,0.284314,16.375,71689.75,55819.5,1.0,With_XRD,False,No,With_XRD_No
559326,42156,1980-01-01,0.181818,-3.25,3263.0,2761.0,1.0,With_XRD,False,No,With_XRD_No
...,...,...,...,...,...,...,...,...,...,...,...
280277,18940,2023-06-01,0.056319,7.69,2604995.19,2447019.12,1.0,With_XRD,True,No,With_XRD_No
1536937,88661,2023-06-01,0.076962,42.8,9920954.4,9276555.96,0.0,Without_XRD,True,No,Without_XRD_No
258098,17826,2023-06-01,-0.440367,1.22,15187.78,27138.82,0.0,Without_XRD,True,No,Without_XRD_No
247970,17134,2023-06-01,-0.309286,0.4835,38867.598,49217.0,0.0,Without_XRD,True,No,Without_XRD_No


## Our Output Functions (Portfolio Returns, CAPM, Tables, Chart)

In [38]:

def format_numeric_cols(data, rounding_map):
    data = data.copy()
    for col, digits in rounding_map.items():
        if col in data.columns and pd.api.types.is_numeric_dtype(data[col]):
            data[col] = data[col].round(digits)
    return data

def calculate_portfolio_returns(data, group_var='research_not'):
    ret_col = 'ret' if 'ret' in data.columns else 'ret_crsp'
    prc_col = 'prc' if 'prc' in data.columns else 'prc_crsp'
    data = data[data[ret_col].notna() & data[prc_col].notna() & data['mktcap'].notna()].copy()
    data = data.sort_values(['permno', 'date'])
    if 'mktcap_lag' not in data.columns:
        data['mktcap_lag'] = data.groupby('permno')['mktcap'].shift(1)
    data['prc_lag'] = data.groupby('permno')[prc_col].shift(1)
    data = data[data['mktcap_lag'].notna() & data['prc_lag'].notna()]
    data['date'] = data['date'] + pd.offsets.MonthEnd(0)
    def calc_vw_return(group):
        total_mktcap = group['mktcap_lag'].sum()
        if total_mktcap == 0:
            return np.nan
        weights = group['mktcap_lag'] / total_mktcap
        return (weights * group[ret_col]).sum()
    portfolio_returns = data.groupby(['date', group_var]).apply(
        lambda x: pd.Series({
            'vw_return': calc_vw_return(x),
            'ew_return': x[ret_col].mean(),
            'total_mktcap': x['mktcap_lag'].sum(),
            'n_stocks': len(x)
        })
    ).reset_index()
    return portfolio_returns.sort_values([group_var, 'date'])

def run_capm_regression(portfolio_returns, market_returns, rf_rate, use_ew_benchmark=False):
    if 'rf' not in market_returns.columns or market_returns['rf'].isna().all():
        raise ValueError("Risk-free rate 'rf' must be present in market_returns (from Fama-French ff.factors_monthly).")
    market_col = 'ewretd' if use_ew_benchmark else 'vwretd'
    portfolio_returns = portfolio_returns.copy()
    portfolio_returns['date'] = portfolio_returns['date'] + pd.offsets.MonthEnd(0)
    cols = ['date', market_col, 'rf']
    market_data = market_returns[cols].copy()
    market_data['date'] = market_data['date'] + pd.offsets.MonthEnd(0)
    market_data = market_data.rename(columns={market_col: 'market_return'})
    reg_data = portfolio_returns.merge(market_data, on='date', how='left')
    reg_data = reg_data[reg_data['portfolio_return'].notna() & reg_data['market_return'].notna() & reg_data['rf'].notna()].copy()
    r = reg_data['rf'].astype(float)
    reg_data['portfolio_excess'] = reg_data['portfolio_return'] - r
    reg_data['market_excess'] = reg_data['market_return'] - r
    reg_data = reg_data[reg_data['portfolio_excess'].notna() & reg_data['market_excess'].notna()].copy()
    if len(reg_data) < 2:
        return None
    market_excess = pd.to_numeric(reg_data['market_excess'], errors='coerce')
    portfolio_excess = pd.to_numeric(reg_data['portfolio_excess'], errors='coerce')
    valid_mask = market_excess.notna() & portfolio_excess.notna()
    market_excess = market_excess[valid_mask].to_numpy(dtype=float)
    portfolio_excess = portfolio_excess[valid_mask].to_numpy(dtype=float)
    if len(market_excess) < 2:
        return None
    X = pd.DataFrame({'market_excess': market_excess})
    X = add_constant(X, has_constant='add')
    model = OLS(portfolio_excess, X).fit()
    n_months = len(portfolio_excess)
    arithmetic_monthly_return = pd.to_numeric(reg_data.loc[valid_mask, 'portfolio_return'], errors='coerce').mean()
    monthly_vol = portfolio_excess.std(ddof=1)
    rf_mean_sample = reg_data.loc[valid_mask, 'rf'].mean()
    params, tvalues, pvalues = model.params, model.tvalues, model.pvalues
    alpha = params.iloc[0]
    beta = params.iloc[1]
    alpha_tstat = tvalues.iloc[0]
    beta_tstat = tvalues.iloc[1]
    alpha_pval = pvalues.iloc[0]
    beta_pval = pvalues.iloc[1]
    ci = model.conf_int(alpha=0.05)
    return pd.Series({
        'arithmetic_monthly_return': arithmetic_monthly_return,
        'annualized_return': (1 + arithmetic_monthly_return) ** 12 - 1,
        'alpha': alpha, 'beta': beta,
        'alpha_tstat': alpha_tstat, 'beta_tstat': beta_tstat,
        'alpha_pval': alpha_pval, 'beta_pval': beta_pval,
        'alpha_lower_ci': ci.iloc[0, 0], 'alpha_upper_ci': ci.iloc[0, 1],
        'beta_lower_ci': ci.iloc[1, 0], 'beta_upper_ci': ci.iloc[1, 1],
        'r_squared': model.rsquared,
        'sharpe_ratio': (arithmetic_monthly_return - rf_mean_sample) / monthly_vol * np.sqrt(12) if monthly_vol > 0 else np.nan,
        'annualized_alpha': alpha * 12,
        'annualized_vol': monthly_vol * np.sqrt(12),
        'n_observations': n_months
    })

def calculate_sharpe_ratio(returns, rf_rate):
    if len(returns) == 0 or returns.isna().all():
        return np.nan
    excess = returns - rf_rate
    sd = returns.std()
    if sd == 0 or pd.isna(sd):
        return np.nan
    return (excess.mean() / sd) * np.sqrt(12)

def calculate_all_portfolio_metrics(portfolio_data, market_data, portfolio_name, rf_rate):
    group_col = portfolio_data.columns[1]
    ew_returns = portfolio_data[['date', 'ew_return', group_col]].copy()
    ew_returns = ew_returns.rename(columns={'ew_return': 'portfolio_return'})
    ew_metrics_list = []
    for group_name, group_data in ew_returns.groupby(group_col):
        result = run_capm_regression(group_data[['date', 'portfolio_return']], market_data, rf_rate, use_ew_benchmark=False)
        if result is not None and len(result) > 0:
            ew_metrics_list.append({**result.to_dict(), group_col: group_name})
    ew_metrics = pd.DataFrame(ew_metrics_list)
    if len(ew_metrics) > 0:
        ew_metrics['weighting'] = 'Equal-Weighted'
    vw_returns = portfolio_data[['date', 'vw_return', group_col]].copy()
    vw_returns = vw_returns.rename(columns={'vw_return': 'portfolio_return'})
    vw_metrics_list = []
    for group_name, group_data in vw_returns.groupby(group_col):
        result = run_capm_regression(group_data[['date', 'portfolio_return']], market_data, rf_rate, use_ew_benchmark=False)
        if result is not None and len(result) > 0:
            vw_metrics_list.append({**result.to_dict(), group_col: group_name})
    vw_metrics = pd.DataFrame(vw_metrics_list)
    if len(vw_metrics) > 0:
        vw_metrics['weighting'] = 'Value-Weighted'
    if len(ew_metrics) > 0 and len(vw_metrics) > 0:
        all_metrics = pd.concat([ew_metrics, vw_metrics], ignore_index=True)
    elif len(ew_metrics) > 0:
        all_metrics = ew_metrics
    elif len(vw_metrics) > 0:
        all_metrics = vw_metrics
    else:
        return pd.DataFrame()
    all_metrics['portfolio'] = all_metrics['weighting'] + '_' + all_metrics[group_col].astype(str)
    all_metrics = all_metrics.drop(columns=[group_col])
    return all_metrics

def format_performance_table(metrics):
    rounding_map = {
        'arithmetic_monthly_return': 6, 'annualized_return': 4, 'alpha': 6, 'beta': 6,
        'alpha_tstat': 6, 'beta_tstat': 2, 'alpha_lower_ci': 6, 'alpha_upper_ci': 6,
        'beta_lower_ci': 6, 'beta_upper_ci': 6, 'r_squared': 6, 'sharpe_ratio': 6
    }
    table = metrics.copy()
    table['Portfolio'] = table['portfolio']
    table = format_numeric_cols(table, rounding_map)
    table['Alpha p-value'] = table['alpha_pval'].apply(lambda x: f"{x:.3e}")
    table['Beta p-value'] = table['beta_pval'].apply(lambda x: f"{x:.3e}")
    return table[['Portfolio', 'arithmetic_monthly_return', 'annualized_return', 'alpha', 'beta', 'alpha_tstat', 'beta_tstat', 'Alpha p-value', 'Beta p-value', 'alpha_lower_ci', 'alpha_upper_ci', 'beta_lower_ci', 'beta_upper_ci', 'r_squared', 'sharpe_ratio']].rename(columns={
        'arithmetic_monthly_return': 'Arithmetic Monthly Return', 'annualized_return': 'Annualized Return',
        'alpha': 'Alpha (Monthly)', 'beta': 'Beta', 'alpha_tstat': 'Alpha t-stat', 'beta_tstat': 'Beta t-stat',
        'alpha_lower_ci': 'Alpha Lower CI (95%)', 'alpha_upper_ci': 'Alpha Upper CI (95%)',
        'beta_lower_ci': 'Beta Lower CI (95%)', 'beta_upper_ci': 'Beta Upper CI (95%)',
        'r_squared': 'R-squared', 'sharpe_ratio': 'Sharpe Ratio'
    })

def create_alpha_table(portfolio_metrics, market_returns, rf_rate_monthly):
    benchmark_sharpe_vw = calculate_sharpe_ratio(market_returns['vwretd'], rf_rate_monthly)
    benchmark_rows = pd.DataFrame({
        'Portfolio': ['CRSP VW Benchmark'],
        'Alpha (monthly %)': ['—'], 't-statistic': ['—'], 'p-value': ['—'],
        'Beta': ['—'], 'R²': ['—'],
        'Sharpe': [round(benchmark_sharpe_vw, 2)],
        'is_significant': [False], 'sort_order': [99]
    })
    def portfolio_display_name(x):
        if 'Value-Weighted' in x and 'With_XRD' in x and '_Yes' in x: return 'R&D Investable VW'
        if 'Value-Weighted' in x and 'With_XRD' in x and '_No' in x: return 'R&D Not invest. VW'
        if 'Value-Weighted' in x and 'Without_XRD' in x and '_Yes' in x: return 'No R&D Investable VW'
        if 'Value-Weighted' in x and 'Without_XRD' in x and '_No' in x: return 'No R&D Not invest. VW'
        if 'Equal-Weighted' in x and 'With_XRD' in x and '_Yes' in x: return 'R&D Investable EW'
        if 'Equal-Weighted' in x and 'With_XRD' in x and '_No' in x: return 'R&D Not invest. EW'
        if 'Equal-Weighted' in x and 'Without_XRD' in x and '_Yes' in x: return 'No R&D Investable EW'
        if 'Equal-Weighted' in x and 'Without_XRD' in x and '_No' in x: return 'No R&D Not invest. EW'
        if 'Equal-Weighted' in x and 'With_XRD' in x: return 'R&D EW'
        if 'Equal-Weighted' in x and 'Without_XRD' in x: return 'No R&D EW'
        if 'Value-Weighted' in x and 'With_XRD' in x: return 'R&D VW'
        if 'Value-Weighted' in x and 'Without_XRD' in x: return 'No R&D VW'
        return x
    alpha_table = portfolio_metrics.copy()
    alpha_table['Portfolio'] = alpha_table['portfolio'].apply(portfolio_display_name)
    alpha_table['alpha_monthly_pct'] = (alpha_table['alpha'] * 100).round(2)
    alpha_table['alpha_star'] = alpha_table['alpha_pval'].apply(lambda x: '***' if x < 0.01 else '**' if x < 0.05 else '*' if x < 0.10 else '')
    alpha_table['Alpha (monthly %)'] = alpha_table['alpha_monthly_pct'].apply(lambda x: f"{x:.2f}") + alpha_table['alpha_star']
    alpha_table['t-statistic'] = alpha_table['alpha_tstat'].round(2)
    alpha_table['p-value'] = alpha_table['alpha_pval'].apply(lambda x: f"{x:.3f}")
    alpha_table['Beta'] = alpha_table['beta'].round(2)
    alpha_table['R²'] = alpha_table['r_squared'].round(2)
    alpha_table['Sharpe'] = alpha_table['sharpe_ratio'].round(2)
    alpha_table['is_significant'] = alpha_table['alpha_pval'] < 0.10
    sort_map = {'R&D VW': 1, 'No R&D VW': 2, 'R&D EW': 3, 'No R&D EW': 4,
                'R&D Investable VW': 1, 'R&D Not invest. VW': 2, 'No R&D Investable VW': 3, 'No R&D Not invest. VW': 4,
                'R&D Investable EW': 5, 'R&D Not invest. EW': 6, 'No R&D Investable EW': 7, 'No R&D Not invest. EW': 8}
    alpha_table['sort_order'] = alpha_table['Portfolio'].map(sort_map).fillna(9)
    alpha_table = alpha_table.sort_values('sort_order')
    alpha_table = alpha_table[['Portfolio', 'Alpha (monthly %)', 't-statistic', 'p-value', 'Beta', 'R²', 'Sharpe', 'is_significant', 'sort_order']].astype(str)
    benchmark_rows = benchmark_rows.astype(str)
    alpha_table = pd.concat([alpha_table, benchmark_rows], ignore_index=True)
    alpha_table = alpha_table.sort_values('sort_order').drop(columns='sort_order')
    return alpha_table

def create_alpha_table_html(alpha_table, custom_css, caption_suffix='',table_type = ""):
    """Create HTML alpha table with same styling as reprex_final_extended (color-coded rows)."""
    alpha_table_display = alpha_table.drop(columns='is_significant', errors='ignore')
    html = f"""
    <!DOCTYPE html>
    <html>
    <head>
    {custom_css}
    </head>
    <body>
    <table class="table table-dark table-striped">
    <caption><b>Alpha Estimation Results: Single-Factor Model</b><br>
    <i>Rp - Rƒ = α + β(R_CRSP_VW - Rƒ)</i>{caption_suffix}<br>
    <i>{table_type}</i></caption>
    <thead>
    <tr>
    <th>Portfolio</th>
    <th>Alpha (monthly %)</th>
    <th>t-statistic</th>
    <th>p-value</th>
    <th>Beta</th>
    <th>R²</th>
    <th>Sharpe</th>
    </tr>
    </thead>
    <tbody>
    """
    for _, row in alpha_table_display.iterrows():
        portfolio = row['Portfolio']
        if portfolio in ['R&D VW', 'R&D EW', 'R&D Investable VW', 'R&D Investable EW']:
            color = '#dd5182'
        elif portfolio in ['No R&D VW', 'No R&D EW', 'No R&D Investable VW', 'No R&D Investable EW']:
            color = '#ffa600'
        elif portfolio in ['R&D Not invest. VW', 'R&D Not invest. EW']:
            color = '#955196'
        elif portfolio in ['No R&D Not invest. VW', 'No R&D Not invest. EW']:
            color = '#003f5c'
        else:
            color = 'white'
        bold = 'font-weight: bold;' if portfolio == 'CRSP VW Benchmark' or 'R&D' in portfolio or 'No R&D' in portfolio else ''
        html += f"""
        <tr style="color: {color}; {bold}">
        <td>{portfolio}</td>
        <td>{row['Alpha (monthly %)']}</td>
        <td>{row['t-statistic']}</td>
        <td>{row['p-value']}</td>
        <td>{row['Beta']}</td>
        <td>{row['R²']}</td>
        <td>{row['Sharpe']}</td>
        </tr>
        """
    html += """
    </tbody>
    </table>
    </body>
    </html>
    """
    return html

custom_css = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Tomorrow:wght@400;600;700&display=swap');
body { font-family: 'Tomorrow', sans-serif; background-color: #1d1d1c; padding: 1rem; margin: 0; color: white; }
.table-dark.table-striped tbody tr:nth-of-type(odd) { background-color: rgba(255, 255, 255, 0.05) !important; }
.table-dark.table-striped tbody tr:nth-of-type(even) { background-color: rgba(0, 0, 0, 0.15) !important; }
.table-dark tbody tr[style*='color'] { background-color: inherit !important; }
.table-dark td, .table-dark th { padding: 0.5rem 0.75rem !important; }
</style>
"""

def create_performance_chart(chart_data, panel_type='vw', colors=None):
    pattern = 'vw_return' if panel_type == 'vw' else 'ew_return'
    benchmark_name = 'CRSP VW Benchmark'
    panel_data = chart_data[chart_data['key'].str.contains(pattern)].copy()
    def key_to_name(x):
        if 'CRSP' in x: return benchmark_name
        if '_Yes' in x: return 'R&D, Investable' if 'With_XRD' in x else 'No R&D, Investable'
        if '_No' in x: return 'R&D, Not investable' if 'With_XRD' in x else 'No R&D, Not investable'
        return 'R&D Portfolio' if 'With_XRD' in x else 'No R&D Portfolio' if 'Without_XRD' in x else x
    panel_data['portfolio_name'] = panel_data['key'].apply(key_to_name)
    fig = go.Figure()
    for portfolio in panel_data['portfolio_name'].unique():
        port_data = panel_data[panel_data['portfolio_name'] == portfolio].sort_values('date')
        color = (colors or {}).get(portfolio, '#888')
        fig.add_trace(go.Scatter(x=port_data['date'], y=port_data['cumvalue'], mode='lines', name=portfolio, line=dict(color=color, width=3)))
    fig.update_layout(title='Cumulative Performance (1980-2022)', xaxis_title='Date', yaxis_title='Cumulative Value ($)',
                      yaxis=dict(rangemode='tozero', tickformat='$.2f'), template='plotly_dark', height=500)
    return fig


## Build Market Returns and Portfolio Metrics

In [39]:
mkt['date'] = pd.to_datetime(mkt['date']).dt.to_period('M').dt.to_timestamp()
rf['date'] = pd.to_datetime(rf['date']).dt.to_period('M').dt.to_timestamp()
if 'rf' not in rf.columns or rf['rf'].isna().all():
    raise ValueError("Risk-free rate 'rf' must exist in Fama-French data (ff.factors_monthly).")
rf['rf'] = pd.to_numeric(rf['rf'], errors='coerce')
market_returns = mkt.merge(rf, on='date', how='left')
if 'rf' not in market_returns.columns or market_returns['rf'].isna().all():
    raise ValueError("Risk-free rate 'rf' must be present in market_returns after merge (from ff.factors_monthly).")
market_returns['vwretd'] = pd.to_numeric(market_returns['vwretd'], errors='coerce')
market_returns['ewretd'] = pd.to_numeric(market_returns['ewretd'], errors='coerce')

portfolio_returns = calculate_portfolio_returns(analysis_data, 'research_not')
portfolio_returns_8 = calculate_portfolio_returns(analysis_data, 'portfolio_id')

rf_rate_monthly = market_returns['rf'].mean()
portfolio_metrics = calculate_all_portfolio_metrics(portfolio_returns, market_returns, 'R&D Portfolio', rf_rate_monthly)
portfolio_metrics_8 = calculate_all_portfolio_metrics(portfolio_returns_8, market_returns, 'R&D × Investability', rf_rate_monthly)

/var/folders/fj/plbn4w9d6fs55651zh_pxsd80000gn/T/ipykernel_73161/2929749822.py:24: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/var/folders/fj/plbn4w9d6fs55651zh_pxsd80000gn/T/ipykernel_73161/2929749822.py:24: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [40]:
portfolio_returns.sort_values('date')

portfolio_returns['new_date'] = portfolio_returns['date'] + pd.offsets.MonthEnd(0)
portfolio_returns

,date,research_not,vw_return,ew_return,total_mktcap,n_stocks,new_date
0,1980-02-29,With_XRD,-0.029445,0.004442,1.813626e+07,92.0,1980-02-29
2,1980-03-31,With_XRD,-0.109543,-0.171458,3.091315e+07,147.0,1980-03-31
4,1980-04-30,With_XRD,0.013823,0.039202,2.980135e+07,182.0,1980-04-30
6,1980-05-31,With_XRD,0.050758,0.069154,4.441125e+08,784.0,1980-05-31
8,1980-06-30,With_XRD,0.027263,0.024925,4.641136e+08,806.0,1980-06-30
...,...,...,...,...,...,...,...
1033,2023-02-28,Without_XRD,-0.035999,-0.037098,9.038696e+09,1022.0,2023-02-28
1035,2023-03-31,Without_XRD,0.009805,-0.052786,8.630802e+09,1011.0,2023-03-31
1037,2023-04-30,Without_XRD,0.013199,-0.035134,2.420190e+09,236.0,2023-04-30
1039,2023-05-31,Without_XRD,-0.035294,-0.026375,1.071317e+09,164.0,2023-05-31


## Outputs: Performance Tables, Alpha Table, Chart

In [41]:
from IPython.display import HTML, display

# Build 3-month lag results if not already computed (e.g. run this cell before the standalone 3m cell)
if 'performance_table_3m' not in dir() or not isinstance(globals().get('performance_table_3m'), pd.DataFrame):
    comp_3m = comp[['permno', 'datadate', 'rd_flag_lag1']].copy()
    comp_3m['available_date'] = pd.to_datetime(comp_3m['datadate']) + pd.Timedelta(days=90)
    comp_3m['available_date'] = comp_3m['available_date'].dt.to_period('M').dt.to_timestamp()
    comp_for_merge_3m = comp_3m[['permno', 'available_date', 'rd_flag_lag1']].dropna(subset=['rd_flag_lag1'])
    comp_for_merge_3m['permno'] = pd.to_numeric(comp_for_merge_3m['permno'], errors='coerce').astype('int64')
    comp_for_merge_3m = comp_for_merge_3m.dropna(subset=['permno'])
    comp_for_merge_3m['available_date'] = pd.to_datetime(comp_for_merge_3m['available_date']).dt.to_period('M').dt.to_timestamp()
    crsp_cols = ['permno', 'date', 'ret', 'prc', 'mktcap', 'mktcap_lag']
    merged_3m = crsp_dates.merge(comp_for_merge_3m, on='permno', how='inner')
    merged_3m = merged_3m[merged_3m['date'] >= merged_3m['available_date']]
    merged_3m = merged_3m.sort_values(['permno', 'date', 'available_date']).drop_duplicates(subset=['permno', 'date'], keep='last')
    merged_3m = merged_3m.merge(crsp[crsp_cols], on=['permno', 'date'], how='inner')
    merged_3m = merged_3m[(merged_3m['available_date'].dt.year >= 1980) & (merged_3m['available_date'].dt.year <= 2022)]
    merged_3m = merged_3m.dropna(subset=['ret', 'mktcap_lag']).query('mktcap_lag > 0')
    analysis_data_3m = merged_3m[['permno', 'date', 'ret', 'prc', 'mktcap', 'mktcap_lag', 'rd_flag_lag1']].copy()
    analysis_data_3m['research_not'] = np.where(analysis_data_3m['rd_flag_lag1'].astype(int) == 1, 'With_XRD', 'Without_XRD')
    analysis_data_3m = analysis_data_3m.sort_values(['permno', 'date'])
    prior_12_3m = analysis_data_3m.groupby('permno')['prc'].transform(lambda x: x.shift(1).rolling(12, min_periods=12).count())
    analysis_data_3m['has_prior_12_prices'] = (prior_12_3m >= 12)
    analysis_data_3m['investability'] = np.where((analysis_data_3m['mktcap_lag'] >= 15e6) & analysis_data_3m['has_prior_12_prices'], 'Yes', 'No')
    analysis_data_3m['portfolio_id'] = analysis_data_3m['research_not'] + '_' + analysis_data_3m['investability']
    portfolio_returns_3m = calculate_portfolio_returns(analysis_data_3m, 'research_not')
    portfolio_returns_8_3m = calculate_portfolio_returns(analysis_data_3m, 'portfolio_id')
    portfolio_metrics_3m = calculate_all_portfolio_metrics(portfolio_returns_3m, market_returns, 'R&D Portfolio', rf_rate_monthly)
    portfolio_metrics_8_3m = calculate_all_portfolio_metrics(portfolio_returns_8_3m, market_returns, 'R&D × Investability', rf_rate_monthly)
    performance_table_3m = format_performance_table(portfolio_metrics_3m)
    performance_table_8_3m = format_performance_table(portfolio_metrics_8_3m)
    alpha_table_8_3m = create_alpha_table(portfolio_metrics_8_3m, market_returns, rf_rate_monthly)

# Build 12-month lag results if not already computed
if 'performance_table_12m' not in dir() or not isinstance(globals().get('performance_table_12m'), pd.DataFrame):
    comp_12m = comp[['permno', 'datadate', 'rd_flag_lag1']].copy()
    comp_12m['available_date'] = pd.to_datetime(comp_12m['datadate']) + pd.Timedelta(days=365)
    comp_12m['available_date'] = comp_12m['available_date'].dt.to_period('M').dt.to_timestamp()
    comp_for_merge_12m = comp_12m[['permno', 'available_date', 'rd_flag_lag1']].dropna(subset=['rd_flag_lag1'])
    comp_for_merge_12m['permno'] = pd.to_numeric(comp_for_merge_12m['permno'], errors='coerce').astype('int64')
    comp_for_merge_12m = comp_for_merge_12m.dropna(subset=['permno'])
    comp_for_merge_12m['available_date'] = pd.to_datetime(comp_for_merge_12m['available_date']).dt.to_period('M').dt.to_timestamp()
    merged_12m = crsp_dates.merge(comp_for_merge_12m, on='permno', how='inner')
    merged_12m = merged_12m[merged_12m['date'] >= merged_12m['available_date']]
    merged_12m = merged_12m.sort_values(['permno', 'date', 'available_date']).drop_duplicates(subset=['permno', 'date'], keep='last')
    merged_12m = merged_12m.merge(crsp[crsp_cols], on=['permno', 'date'], how='inner')
    merged_12m = merged_12m[(merged_12m['available_date'].dt.year >= 1980) & (merged_12m['available_date'].dt.year <= 2022)]
    merged_12m = merged_12m.dropna(subset=['ret', 'mktcap_lag']).query('mktcap_lag > 0')
    analysis_data_12m = merged_12m[['permno', 'date', 'ret', 'prc', 'mktcap', 'mktcap_lag', 'rd_flag_lag1']].copy()
    analysis_data_12m['research_not'] = np.where(analysis_data_12m['rd_flag_lag1'].astype(int) == 1, 'With_XRD', 'Without_XRD')
    analysis_data_12m = analysis_data_12m.sort_values(['permno', 'date'])
    prior_12_12m = analysis_data_12m.groupby('permno')['prc'].transform(lambda x: x.shift(1).rolling(12, min_periods=12).count())
    analysis_data_12m['has_prior_12_prices'] = (prior_12_12m >= 12)
    analysis_data_12m['investability'] = np.where((analysis_data_12m['mktcap_lag'] >= 15e6) & analysis_data_12m['has_prior_12_prices'], 'Yes', 'No')
    analysis_data_12m['portfolio_id'] = analysis_data_12m['research_not'] + '_' + analysis_data_12m['investability']
    portfolio_returns_12m = calculate_portfolio_returns(analysis_data_12m, 'research_not')
    portfolio_returns_8_12m = calculate_portfolio_returns(analysis_data_12m, 'portfolio_id')
    portfolio_metrics_12m = calculate_all_portfolio_metrics(portfolio_returns_12m, market_returns, 'R&D Portfolio', rf_rate_monthly)
    portfolio_metrics_8_12m = calculate_all_portfolio_metrics(portfolio_returns_8_12m, market_returns, 'R&D × Investability', rf_rate_monthly)
    performance_table_12m = format_performance_table(portfolio_metrics_12m)
    performance_table_8_12m = format_performance_table(portfolio_metrics_8_12m)
    alpha_table_8_12m = create_alpha_table(portfolio_metrics_8_12m, market_returns, rf_rate_monthly)

TEST_PERIOD = '1980–2022'

# ---- 3-month lag tables (same color-coding as reprex_final_extended) ----
print('\n' + '='*70)
print('Reporting lag: 3 months | Test period:', TEST_PERIOD)
print('='*70)
print('\nPerformance Table (2 portfolios):')
print(performance_table_3m.to_string())
print('\nPerformance Table — 8 portfolios (R&D × Investability):')
print(performance_table_8_3m.to_string())
print('\nAlpha Estimation Results (3-month lag):')
display(HTML(create_alpha_table_html(alpha_table_8_3m, custom_css, caption_suffix=f'<br>Test period: {TEST_PERIOD}',table_type = "3-month lag")))

# ---- 4-month lag tables ----
performance_table = format_performance_table(portfolio_metrics)
performance_table_8 = format_performance_table(portfolio_metrics_8)
print('\n' + '='*70)
print('Reporting lag: 4 months | Test period:', TEST_PERIOD)
print('='*70)
print('\nPerformance Table (2 portfolios):')
print(performance_table.to_string())
print('\nPerformance Table — 8 portfolios (R&D × Investability):')
print(performance_table_8.to_string())
print('\nAlpha Estimation Results (4-month lag):')
alpha_table_8 = create_alpha_table(portfolio_metrics_8, market_returns, rf_rate_monthly)
display(HTML(create_alpha_table_html(alpha_table_8, custom_css, caption_suffix=f'<br>Test period: {TEST_PERIOD}',table_type = "4-month lag")))

# ---- 12-month lag tables ----
print('\n' + '='*70)
print('Reporting lag: 12 months | Test period:', TEST_PERIOD)
print('='*70)
print('\nPerformance Table (2 portfolios):')
print(performance_table_12m.to_string())
print('\nPerformance Table — 8 portfolios (R&D × Investability):')
print(performance_table_8_12m.to_string())
print('\nAlpha Estimation Results (12-month lag):')
display(HTML(create_alpha_table_html(alpha_table_8_12m, custom_css, caption_suffix=f'<br>Test period: {TEST_PERIOD}', table_type="12-month lag")))

# Chart data (8 portfolios + benchmark)
chart_data_8 = portfolio_returns_8.melt(id_vars=['date', 'portfolio_id'], value_vars=['ew_return', 'vw_return'], var_name='weighting', value_name='value')
chart_data_8['key'] = chart_data_8['weighting'] + '_' + chart_data_8['portfolio_id']
market_vw = market_returns.copy()
market_vw['date'] = market_vw['date'] + pd.offsets.MonthEnd(0)
market_vw['key'] = 'vw_return_CRSP_VW_Benchmark'
market_vw['value'] = market_vw['vwretd']
chart_data = pd.concat([chart_data_8[['date', 'key', 'value']], market_vw[['date', 'key', 'value']]], ignore_index=True)
# Common start date so all series align (earliest date where every key has data)
common_start = chart_data.groupby('key')['date'].min().max()
chart_data = chart_data[chart_data['date'] >= common_start].copy()
chart_data['cumvalue'] = (1.0 + pd.to_numeric(chart_data['value'], errors='coerce')).groupby(chart_data['key']).cumprod()
chart_data['cumvalue'] = chart_data.groupby('key')['cumvalue'].transform(lambda x: x / x.iloc[0])

colors = {'CRSP VW Benchmark': '#58508d', 'R&D, Investable': '#dd5182', 'No R&D, Investable': '#ffa600', 'R&D, Not investable': '#955196', 'No R&D, Not investable': '#003f5c'}
fig_vw = create_performance_chart(chart_data, panel_type='vw', colors=colors)
fig_vw.show()

db.close()


Reporting lag: 3 months | Test period: 1980–2022

Performance Table (2 portfolios):
                    Portfolio  Arithmetic Monthly Return  Annualized Return  Alpha (Monthly)      Beta  Alpha t-stat  Beta t-stat Alpha p-value Beta p-value  Alpha Lower CI (95%)  Alpha Upper CI (95%)  Beta Lower CI (95%)  Beta Upper CI (95%)  R-squared  Sharpe Ratio
0     Equal-Weighted_With_XRD                   0.013400             0.1732         0.001516  1.255947      1.991569        88.81     4.694e-02   5.083e-316              0.000021              0.003012             1.228165             1.283729   0.938262      0.507515
1  Equal-Weighted_Without_XRD                   0.011541             0.1476         0.001131  1.041678      1.764437        87.48     7.825e-02   8.068e-313             -0.000128              0.002390             1.018284             1.065072   0.936484      0.499422
2     Value-Weighted_With_XRD                   0.010875             0.1386         0.000636  1.068895      1.0

Portfolio,Alpha (monthly %),t-statistic,p-value,Beta,R²,Sharpe
R&D Investable VW,0.13,1.44,0.151,0.97,0.83,0.55
R&D Not invest. VW,0.01,0.06,0.952,1.21,0.88,0.47
No R&D Investable VW,0.10,0.84,0.404,0.87,0.73,0.51
No R&D Not invest. VW,0.01,0.16,0.871,1.01,0.87,0.48
R&D Investable EW,0.27**,2.12,0.034,0.7,0.63,0.55
R&D Not invest. EW,0.14*,1.75,0.081,1.28,0.93,0.5
No R&D Investable EW,0.27*,1.69,0.093,0.61,0.49,0.51
No R&D Not invest. EW,0.10,1.51,0.131,1.06,0.93,0.49
CRSP VW Benchmark,—,—,—,—,—,0.42



Reporting lag: 4 months | Test period: 1980–2022

Performance Table (2 portfolios):
                    Portfolio  Arithmetic Monthly Return  Annualized Return  Alpha (Monthly)      Beta  Alpha t-stat  Beta t-stat Alpha p-value Beta p-value  Alpha Lower CI (95%)  Alpha Upper CI (95%)  Beta Lower CI (95%)  Beta Upper CI (95%)  R-squared  Sharpe Ratio
0     Equal-Weighted_With_XRD                   0.013448             0.1739         0.001931  1.264135      1.120860        33.51     2.629e-01   6.847e-132             -0.001454              0.005316             1.190021             1.338250   0.683887      0.510889
1  Equal-Weighted_Without_XRD                   0.011473             0.1467         0.001147  1.082030      0.855162        36.84     3.929e-01   6.308e-147             -0.001488              0.003783             1.024324             1.139737   0.723338      0.494979
2     Value-Weighted_With_XRD                   0.010746             0.1369         0.000518  1.067127      0.8

Portfolio,Alpha (monthly %),t-statistic,p-value,Beta,R²,Sharpe
R&D Investable VW,0.12,1.43,0.153,0.97,0.84,0.55
R&D Not invest. VW,0.01,0.13,0.895,1.21,0.88,0.48
No R&D Investable VW,0.10,0.91,0.363,0.87,0.73,0.51
No R&D Not invest. VW,0.01,0.15,0.884,1.01,0.87,0.48
R&D Investable EW,0.11,1.51,0.131,0.98,0.88,0.55
R&D Not invest. EW,0.19,1.05,0.292,1.28,0.67,0.5
No R&D Investable EW,0.08,0.78,0.437,0.9,0.78,0.51
No R&D Not invest. EW,0.11,0.78,0.436,1.09,0.71,0.49
CRSP VW Benchmark,—,—,—,—,—,0.42



Reporting lag: 12 months | Test period: 1980–2022

Performance Table (2 portfolios):
                    Portfolio  Arithmetic Monthly Return  Annualized Return  Alpha (Monthly)      Beta  Alpha t-stat  Beta t-stat Alpha p-value Beta p-value  Alpha Lower CI (95%)  Alpha Upper CI (95%)  Beta Lower CI (95%)  Beta Upper CI (95%)  R-squared  Sharpe Ratio
0     Equal-Weighted_With_XRD                   0.013899             0.1802         0.001927  1.248693      2.654340        91.85     8.193e-03   4.585e-321              0.000501              0.003353             1.221984             1.275403   0.942568      0.544635
1  Equal-Weighted_Without_XRD                   0.012219             0.1569         0.001774  1.031559      2.648118        82.22     8.343e-03   6.171e-298              0.000458              0.003090             1.006912             1.056206   0.929346      0.551929
2     Value-Weighted_With_XRD                   0.011157             0.1424         0.000843  1.073890      1.

Portfolio,Alpha (monthly %),t-statistic,p-value,Beta,R²,Sharpe
R&D Investable VW,0.10,1.22,0.223,0.98,0.85,0.56
R&D Not invest. VW,0.09,0.96,0.337,1.21,0.88,0.53
No R&D Investable VW,0.10,0.91,0.364,0.87,0.73,0.51
No R&D Not invest. VW,0.07,0.88,0.382,1.01,0.87,0.53
R&D Investable EW,0.28**,2.16,0.032,0.71,0.64,0.57
R&D Not invest. EW,0.18**,2.36,0.019,1.27,0.94,0.54
No R&D Investable EW,0.26,1.64,0.102,0.61,0.49,0.5
No R&D Not invest. EW,0.17**,2.39,0.017,1.05,0.92,0.54
CRSP VW Benchmark,—,—,—,—,—,0.42


## 3-month lag pipeline (for comparison tables)

Run the same merge and metrics with a **3-month (90-day) reporting lag** so we can display tables for both 3- and 4-month lags with headings that match the test period (1980–2022).

In [42]:
# 3-month (90-day) reporting lag: build merged_3m, analysis_data_3m, portfolio metrics and tables
comp_3m = comp[['permno', 'datadate', 'rd_flag_lag1']].copy()
comp_3m['available_date'] = pd.to_datetime(comp_3m['datadate']) + pd.Timedelta(days=90)
comp_3m['available_date'] = comp_3m['available_date'].dt.to_period('M').dt.to_timestamp()

comp_for_merge_3m = comp_3m[['permno', 'available_date', 'rd_flag_lag1']].dropna(subset=['rd_flag_lag1'])
comp_for_merge_3m['permno'] = pd.to_numeric(comp_for_merge_3m['permno'], errors='coerce').astype('int64')
comp_for_merge_3m = comp_for_merge_3m.dropna(subset=['permno'])
comp_for_merge_3m['available_date'] = pd.to_datetime(comp_for_merge_3m['available_date']).dt.to_period('M').dt.to_timestamp()

crsp_cols = ['permno', 'date', 'ret', 'prc', 'mktcap', 'mktcap_lag']
merged_3m = crsp_dates.merge(comp_for_merge_3m, on='permno', how='inner')
merged_3m = merged_3m[merged_3m['date'] >= merged_3m['available_date']]
merged_3m = merged_3m.sort_values(['permno', 'date', 'available_date'])
merged_3m = merged_3m.drop_duplicates(subset=['permno', 'date'], keep='last')
merged_3m = merged_3m.merge(crsp[crsp_cols], on=['permno', 'date'], how='inner')
merged_3m = merged_3m[(merged_3m['available_date'].dt.year >= 1980) & (merged_3m['available_date'].dt.year <= 2022)]
merged_3m = merged_3m.dropna(subset=['ret', 'mktcap_lag'])
merged_3m = merged_3m[merged_3m['mktcap_lag'] > 0]

analysis_data_3m = merged_3m[['permno', 'date', 'ret', 'prc', 'mktcap', 'mktcap_lag', 'rd_flag_lag1']].copy()
analysis_data_3m['research_not'] = np.where(analysis_data_3m['rd_flag_lag1'].astype(int) == 1, 'With_XRD', 'Without_XRD')
analysis_data_3m = analysis_data_3m.sort_values(['permno', 'date'])
prior_12_prc_count_3m = analysis_data_3m.groupby('permno')['prc'].transform(lambda x: x.shift(1).rolling(12, min_periods=12).count())
analysis_data_3m['has_prior_12_prices'] = (prior_12_prc_count_3m >= 12)
size_ok_3m = (analysis_data_3m['mktcap_lag'] >= 15e6) & analysis_data_3m['mktcap_lag'].notna()
analysis_data_3m['investability'] = np.where(size_ok_3m & analysis_data_3m['has_prior_12_prices'], 'Yes', 'No')
analysis_data_3m['portfolio_id'] = analysis_data_3m['research_not'] + '_' + analysis_data_3m['investability']

portfolio_returns_3m = calculate_portfolio_returns(analysis_data_3m, 'research_not')
portfolio_returns_8_3m = calculate_portfolio_returns(analysis_data_3m, 'portfolio_id')
portfolio_metrics_3m = calculate_all_portfolio_metrics(portfolio_returns_3m, market_returns, 'R&D Portfolio', rf_rate_monthly)
portfolio_metrics_8_3m = calculate_all_portfolio_metrics(portfolio_returns_8_3m, market_returns, 'R&D × Investability', rf_rate_monthly)

performance_table_3m = format_performance_table(portfolio_metrics_3m)
performance_table_8_3m = format_performance_table(portfolio_metrics_8_3m)
alpha_table_8_3m = create_alpha_table(portfolio_metrics_8_3m, market_returns, rf_rate_monthly)

/var/folders/fj/plbn4w9d6fs55651zh_pxsd80000gn/T/ipykernel_73161/2929749822.py:24: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/var/folders/fj/plbn4w9d6fs55651zh_pxsd80000gn/T/ipykernel_73161/2929749822.py:24: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [44]:
performance_table.columns

performance_table[['Portfolio','Alpha (Monthly)','Alpha t-stat','Sharpe Ratio','Alpha p-value','Beta','R-squared']].to_clipboard()